In [189]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial

import random
import numpy as np

import torch


from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import test_nphases_mcts_search_v55
from core import test_nphases_mcts_search_v53
from core import test_nphases_mcts_search_v15
from core import test_nphases_mcts_search_v13
from core import test_nphases_mcts_search_v35
from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.2
# llm_vllm = LLM(
#     model = llm_dir,
#     tensor_parallel_size=1,
#     gpu_memory_utilization = 0.7,  # Utilize 50% of GPU memory
#     # enable_prefix_caching=True,  # V100 doesn't support enable_prefix_caching 
#     # enable_chunked_prefill=False, # and enable_chunked_prefill
#     max_model_len = 5000,
#     dtype = "float16",
#     seed = config.seed)

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)

print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))



INFO 09-04 11:21:40 [config.py:823] This model supports multiple tasks: {'classify', 'reward', 'embed', 'score', 'generate'}. Defaulting to 'generate'.
WARNING 09-04 11:21:40 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 09-04 11:21:40 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 09-04 11:21:40 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-04 11:21:40 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-04 11:21:43 [default_loader.py:272] Loading weights took 1.31 seconds
INFO 09-04 11:21:44 [model_runner.py:1203] Model loading took 2.3185 GiB and 1.430868 seconds
INFO 09-04 11:21:45 [worker.py:294] Memory profiling takes 0.54 seconds
INFO 09-04 11:21:45 [worker.py:294] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.20) = 6.35GiB
INFO 09-04 11:21:45 [worker.py:294] model weights take 2.32GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 2.75GiB.
INFO 09-04 11:21:45 [executor_base.py:113] # cuda blocks: 5631, # CPU blocks: 32768
INFO 09-04 11:21:45 [executor_base.py:118] Maximum concurrency for 5000 tokens per request: 18.02x
INFO 09-04 11:21:53 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 8.75 seconds
#--- memory: 5.07647705078125


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    task="embed",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

WARNING 09-04 11:21:53 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 09-04 11:21:53 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-04 11:21:53 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_pr

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-04 11:21:56 [default_loader.py:272] Loading weights took 1.51 seconds
INFO 09-04 11:21:56 [model_runner.py:1203] Model loading took 2.3029 GiB and 1.552391 seconds
#--- memory: 7.379390716552734


In [6]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 22.336918354034424


In [ ]:
def level_order_rec(cur_node, level, level_thres, res):
    if cur_node is None:
        return 

    if level > level_thres:
        return
    
    if len(res) <= level:
        res.append([])

    res[level].append(cur_node.visit_count())
    for child_node in cur_node.children:
        level_order_rec(child_node, level+1, level_thres, res)

def level_order(root, level_thres):
    res = []
    level_order_rec(root, 0, level_thres, res)
    return res

In [7]:
stop

NameError: name 'stop' is not defined

In [8]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [9]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 40            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0
config.date_string = "Aug 1 2025"

# mcts parameter

config.num_batches = 2
config.step_budget = config.num_batches*config.max_depths 
config.num_phases = config.step_budget

config.lam = 10 
config.normalize_embeds = True

config.cpuct_root = 0
config.cpuct_leaf = 0
config.cpuct = 2
config.ds_beta = 1.0
config.ds_alpha = 100.0
config.use_ppl = True

# config.version = "v51"



In [ ]:
print(config.date_string)

In [ ]:
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

In [176]:
level = 3                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
num_questions = 1
num_trials = 1
print(f"num_questions = {num_questions}")
print(f"num_trials = {num_trials}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 0
batch_of_questions = [data_by_levels[level][q_idx]['problem']]

num_questions = 1
num_trials = 1


In [177]:
importlib.reload(test_nphases_mcts_search_v15)

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v15.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_nphases_mcts_search_v15.mcts_search(question, agent, config, llm_vllm, prm)



-> p = 0
selected_node = 0

-> d = 0
0.1
   q-value = 0.6514
   u-value = 0.0000
   puct = 0.6514
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.9175
   u-value = 0.0000
   puct = 0.9175
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.7881
   u-value = 0.0000
   puct = 0.7881
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.4
   q-value = 0.1824
   u-value = 0.0000
   puct = 0.1824
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.2
selected_node = 0.2
step_cnt = 1

-> d = 1
0.2.1
   q-value = 0.9561
   u-value = 0.0000
   puct = 0.9561
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.2
   q-value = 0.8989
   u-value = 0.0000
   puct = 0.8989
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.3
   q-value = 0.7798
   u-value = 0.0000
   puct = 0.7798
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.4
   q-value 

In [178]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 13
nvisits = [[13], [2, 5, 5, 5], [1, 2, 2, 2, 2, 2, 2, 2, 2, 3, 1, 3, 1], [2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 3, 1, 1, 1, 1, 1, 3]]
total_nphases = 11
c_nphases = [0, 1, 2, 3, 4, 5, 5, 6, 7, 8, 9, 9, 10] | 5.3 (±0.9)
c_ndepths = [8, 6, 6, 2, 10, 11, 12, 6, 10, 7, 8, 9, 10] | 8.1 (±0.7)


In [179]:
importlib.reload(test_nphases_mcts_search_v13)

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v13.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_nphases_mcts_search_v13.mcts_search(question, agent, config, llm_vllm, prm)




-> p = 0
selected_node = 0

-> d = 0
0.1
   q-value = 0.6514
   u-value = 0.0000
   puct = 0.6514
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.9175
   u-value = 0.0000
   puct = 0.9175
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.7881
   u-value = 0.0000
   puct = 0.7881
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.4
   q-value = 0.1824
   u-value = 0.0000
   puct = 0.1824
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.2
selected_node = 0.2
step_cnt = 1

-> d = 1
0.2.1
   q-value = 0.9561
   u-value = 0.0000
   puct = 0.9561
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.2
   q-value = 0.8989
   u-value = 0.0000
   puct = 0.8989
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.3
   q-value = 0.7798
   u-value = 0.0000
   puct = 0.7798
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.4
   q-value 

In [180]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 13
nvisits = [[13], [2, 5, 5, 5], [1, 2, 1, 1, 2, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 3, 1, 1, 3, 1, 1, 1], [2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 3, 1, 1, 1, 1, 3, 1]]
total_nphases = 11
c_nphases = [0, 1, 2, 3, 4, 5, 5, 6, 7, 8, 9, 9, 10] | 5.3 (±0.9)
c_ndepths = [8, 6, 8, 2, 17, 7, 8, 6, 8, 8, 6, 7, 8] | 7.6 (±0.9)


In [181]:
importlib.reload(test_nphases_mcts_search_v55)

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v55.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_nphases_mcts_search_v55.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)


-> p = 0
selected_node = 0

-> d = 0
0.1
   q-value = 0.6514
   u-value = 0.0000
   puct = 0.6514
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.9175
   u-value = 0.0000
   puct = 0.9175
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.7881
   u-value = 0.0000
   puct = 0.7881
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.4
   q-value = 0.1824
   u-value = 0.0000
   puct = 0.1824
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.2
selected_node = 0.2
step_cnt = 1

-> d = 1
0.2.1
   q-value = 0.9561
   u-value = 0.0000
   puct = 0.9561
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.2
   q-value = 0.8989
   u-value = 0.0000
   puct = 0.8989
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.3
   q-value = 0.7798
   u-value = 0.0000
   puct = 0.7798
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.4
   q-value 

In [182]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 13
nvisits = [[13], [2, 4, 6, 5], [1, 3, 2, 1, 1, 3, 4, 1, 1, 4, 1, 2, 1], [2, 1, 1, 2, 1, 1, 1, 2, 1, 1, 2, 2, 1, 1, 4, 1, 1, 4, 1, 1, 2, 1, 1, 1]]
total_nphases = 8
c_nphases = [0, 1, 2, 3, 3, 3, 4, 5, 6, 6, 6, 7, 8] | 4.2 (±0.7)
c_ndepths = [8, 2, 7, 7, 8, 9, 7, 9, 7, 9, 10, 6, 12] | 7.8 (±0.7)


In [183]:
importlib.reload(test_nphases_mcts_search_v53)

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v53.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_nphases_mcts_search_v53.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)


-> p = 0
selected_node = 0

-> d = 0
0.1
   q-value = 0.6514
   u-value = 0.0000
   puct = 0.6514
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.9175
   u-value = 0.0000
   puct = 0.9175
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.7881
   u-value = 0.0000
   puct = 0.7881
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.4
   q-value = 0.1824
   u-value = 0.0000
   puct = 0.1824
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.2
selected_node = 0.2
step_cnt = 1

-> d = 1
0.2.1
   q-value = 0.9561
   u-value = 0.0000
   puct = 0.9561
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.2
   q-value = 0.8989
   u-value = 0.0000
   puct = 0.8989
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.3
   q-value = 0.7798
   u-value = 0.0000
   puct = 0.7798
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2.4
   q-value 

In [184]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 11
nvisits = [[15], [2, 6, 5, 6], [1, 3, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 6], [2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 4, 1, 1, 3, 1, 1, 2, 1, 1, 2, 1, 1, 1, 6, 1, 1, 1]]
total_nphases = 7
c_nphases = [0, 1, 1, 1, 2, 3, 3, 4, 5, 6, 7] | 3.0 (±0.7)
c_ndepths = [8, 6, 7, 7, 2, 6, 7, 8, 4, 7, 8] | 6.4 (±0.6)


In [260]:
level = 3                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
num_questions = 1
num_trials = 1
print(f"num_questions = {num_questions}")
print(f"num_trials = {num_trials}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 0
batch_of_questions = [data_by_levels[level][q_idx]['problem']]

num_questions = 1
num_trials = 1


In [270]:
importlib.reload(test_nphases_mcts_search_v55)

config.lam = 10
config.ds_alpha = 1.0

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v55.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_nphases_mcts_search_v55.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)


-> p = 0
selected_node = 0

-> d = 0
0.1
   q-value = 0.3923
   u-value = 0.0000
   puct = 0.3923
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.3311
   u-value = 0.0000
   puct = 0.3311
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.5850
   u-value = 0.0000
   puct = 0.5850
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.4
   q-value = 0.4036
   u-value = 0.0000
   puct = 0.4036
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.3
selected_node = 0.3
step_cnt = 1

-> d = 1
0.3.1
   q-value = 0.7607
   u-value = 0.0000
   puct = 0.7607
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
selected_child = 0.3.1
selected_node = 0.3.1
step_cnt = 2

-> d = 2
0.3.1.1
   q-value = 0.9473
   u-value = 0.0000
   puct = 0.9473
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
selected_child = 0.3.1.1
selected_node = 0.3.1.1
step_cnt = 3

-> d = 3
0.

In [268]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 20
nvisits = [[20], [6, 5, 8, 5], [6, 3, 3, 8, 2, 3, 2], [2, 3, 3, 2, 2, 3, 8, 2, 2, 2, 1, 1, 2]]
total_nphases = 17
c_nphases = [0, 0, 1, 2, 3, 4, 4, 5, 6, 7, 8, 8, 9, 9, 10, 11, 12, 13, 14, 15] | 7.0 (±1.0)
c_ndepths = [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 5, 5, 6, 5, 5] | 5.1 (±0.1)


In [263]:
importlib.reload(test_nphases_mcts_search_v35)

np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v35.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, p, ndepths_arr = \
        test_nphases_mcts_search_v35.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)


-> p = 0
selected_node = 0

-> d = 0
0.1
   q-value = 0.3923
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.3311
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.5850
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.4
   q-value = 0.4036
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
vals = [32.01511004 31.95383516 32.20773716 32.02634262]
beta*q_values = [0.39233398 0.33105469 0.58496094 0.40356445]
alpha*q_diversity = [31.62277606 31.62278047 31.62277622 31.62277817]
candidate_idxes = [3]
best_idx = 3
selected_child = 0.3
selected_node = 0.3
step_cnt = 1

-> d = 1
0.3.1
   q-value = 0.7607
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
vals = [32.38352022]
beta*q_values = [0.76074219]
alpha*q_diversity = [31.62277804]
candidate_idxes = [1]
best_idx = 1
selected_child = 0.3.1
selected_node = 0.3.1
step_cnt = 2

-> d = 2
0.3.1.1
   q-value = 0.9473
   nvisit = 1.00


In [214]:
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

ncomps = 17
nvisits = [[17], [2, 5, 4, 10], [1, 4, 2, 1, 1, 2, 2, 2, 1, 10, 1, 1, 1], [2, 1, 1, 3, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 10]]
total_nphases = 10
c_nphases = [0, 1, 2, 3, 4, 4, 5, 6, 6, 6, 6, 6, 7, 7, 8, 8, 9] | 5.2 (±0.6)
c_ndepths = [8, 2, 9, 6, 13, 15, 7, 14, 19, 20, 21, 22, 6, 7, 10, 14, 7] | 11.8 (±1.5)
